## Cellpose-SAM: superhuman generalization for cellular segmentation

Marius Pachitariu, Michael Rariden, Carsen Stringer

[paper](https://www.biorxiv.org/content/10.1101/2025.04.28.651001v1) | [code](https://github.com/MouseLand/cellpose)

This notebook shows how to process your own 2D or 3D images, saved on Google Drive.

This notebook is adapted from the notebook by Pradeep Rajasekhar, inspired by the [ZeroCostDL4Mic notebook series](https://github.com/HenriquesLab/ZeroCostDL4Mic/wiki).

### Install Cellpose-SAM


In [ ]:
# !pip install git+https://www.github.com/mouseland/cellpose.git

Check GPU and instantiate model - will download weights.

In [ ]:
import numpy as np
from cellpose import models, core, io, plot
from pathlib import Path
from tqdm import trange
import matplotlib.pyplot as plt
from natsort import natsorted
import torch
import cv2

io.logger_setup() # run this to get printing of progress

print('Running on GPU: ', torch.cuda.get_device_name(0))

#Check if colab notebook instance has GPU access
if core.use_gpu()==False:
  raise ImportError("No GPU access, change your runtime")

model = models.CellposeModel(gpu=True)

In [ ]:
# *** change to your google drive folder path ***
dir = "../data/Adults_training_Raw"
dir = Path(dir)
if not dir.exists():
  raise FileNotFoundError("directory does not exist")

# *** change to your image extension ***
image_ext = ".tif"

# list all files
files = natsorted([f for f in dir.glob("*"+image_ext) if "_masks" not in f.name and "_flows" not in f.name])

if(len(files)==0):
  raise FileNotFoundError("no image files found, did you specify the correct folder and extension?")
else:
  print(f"{len(files)} images in folder:")

for f in files:
  print(f.name)

## Helper functions

In [ ]:
from skimage.segmentation import find_boundaries
def outline_view_skimage(img0, maski, color=[1, 0, 0], mode="inner"):
    """
    Generates a red outline overlay onto the image.

    Args:
        img0 (numpy.ndarray): The input image.
        maski (numpy.ndarray): The mask representing the region of interest.
        color (list, optional): The color of the outline overlay. Defaults to [1, 0, 0] (red).
        mode (str, optional): The mode for generating the outline. Defaults to "inner".

    Returns:
        numpy.ndarray: The image with the red outline overlay.

    """
    if img0.ndim == 2:
        img0 = np.stack([img0] * 3, axis=-1)
    elif img0.ndim != 3:
        raise ValueError("img0 not right size (must have ndim 2 or 3)")

    outlines = find_boundaries(maski, mode=mode)
    outY, outX = np.nonzero(outlines)
    imgout = img0.copy()
    imgout[outY, outX] = np.array(color)

    return imgout, outlines

## Run Cellpose-SAM on one image in folder

Here are some of the parameters you can change:

* ***flow_threshold*** is  the  maximum  allowed  error  of  the  flows  for  each  mask.   The  default  is 0.4.
    *  **Increase** this threshold if cellpose is not returning as many masks as you’d expect (or turn off completely with 0.0)
    *   **Decrease** this threshold if cellpose is returning too many ill-shaped masks.

* ***cellprob_threshold*** determines proability that a detected object is a cell.   The  default  is 0.0.
    *   **Decrease** this threshold if cellpose is not returning as many masks as you’d expect or if masks are too small
    *   **Increase** this threshold if cellpose is returning too many masks esp from dull/dim areas.

* ***tile_norm_blocksize*** determines the size of blocks used for normalizing the image. The default is 0, which means the entire image is normalized together.
  You may want to change this to 100-200 pixels if you have very inhomogeneous brightness across your image.



In [ ]:
# take only images start with TIFF_AH_160
files = [f for f in files if f.name.startswith('TIFF_AH_157')]
files

In [ ]:
img_0 = io.imread(files[0])
print('img_0', img_0.max(), img_0.min())
print('type', img_0.dtype)
img_1 = io.imread(files[1])
print('img_1', img_1.max(), img_1.min())
print('type', img_1.dtype)
img_2 = io.imread(files[2])
print('img_2', img_2.max(), img_2.min())
print('type', img_2.dtype)
img_3 = io.imread(files[3])
print('img_3', img_3.max(), img_3.min())
print('type', img_3.dtype)
img_4 = io.imread(files[4])
print('img_4', img_4.max(), img_4.min())
print('type', img_4.dtype)
img_5 = io.imread(files[5])
print('img_5', img_5.max(), img_5.min())
print('type', img_5.dtype)
img_6 = np.ones((100, 100, 3)) * 255

In [ ]:
image_no = 2
img_ = io.imread(files[image_no])

print(f'your image has shape: {img_.shape}. Assuming channel dimension is last with {img_.shape[-1]} channels')

In [ ]:
plt.imshow(img_)
plt.show()

In [ ]:
# print(img_.shape)

# plt.imshow(img_[3654:3690,5432:5933,:])
# plt.show()

In [ ]:
# # Split the image in to 4 patches
# patches = np.array_split(img_, 2, axis=0)
# for patch in patches:
#     plt.imshow(patch)
#     plt.show()

# all_patches = []
# # # split the patches in to 2x2 patches
# for i in range(2):
#     img_patche = patches[i]
#     patches = np.array_split(img_, 2, axis=0)


In [ ]:
def normalize99(img):
    X = img.copy()
    X = (X - np.percentile(X, 1)) / (1e-10 + np.percentile(X, 99) - np.percentile(X, 1))
    return X

def image_resize(img, resize=400):
    ny,nx = img.shape[:2]
    print('before resize', ny, nx)
    if np.array(img.shape).max() > resize:
        if ny>nx:
            nx = int(nx/ny * resize)
            ny = resize
        else:
            ny = int(ny/nx * resize)
            nx = resize
        shape = (nx,ny)
        img = cv2.resize(img, shape)
    print('after resize', img.shape)
    if img.dtype == np.uint16:
        img = img.astype(np.uint16)
    else:
        img = img.astype(np.uint8)
    return img

img = img_.copy()
img = image_resize(img, resize = 1000)

img = normalize99(img)
img = np.clip(img, 0, 1)

img[:,:,0] = np.where(img[:,:,0] > 0.75, 1, img[:,:,0]) # comment out for nucleus segmentation
# make as a grid of images
plt.figure(figsize=(30, 20))
plt.subplot(1, 4, 1)
plt.imshow(img[:,:,0], cmap='gray')
plt.subplot(1, 4, 2)
plt.imshow(img[:,:,1], cmap='gray')
plt.subplot(1, 4, 3)
plt.imshow(img[:,:,2], cmap='gray')
plt.subplot(1, 4, 4)
plt.imshow(img)
plt.show()

# Save as a tiff file
# io.imsave('img.tif', img)


### Channel Selection: 

- Use the dropdowns below to select the _zero-indexed_ channels of your image to segment. The order does not matter. Remember to rerun the cell after you edit the dropdowns.

- If you have a histological image taken in brightfield, you don't need to adjust the channels.

- If you have a fluroescent image with multiple stains, you should choose one channel with a cytoplasm/membrane stain, one channel with a nuclear stain, and set the third channel to `None`. Choosing multiple channels may produce segmentaiton of all the structures in the image. If you have retrained the model on your data with a thrid stain (described below), you can run segmentation with all channels. 

In [ ]:
# for Membrane segmentation
first_channel = 'None' # @param ['None', 0, 1, 2, 3, 4, 5]
second_channel = '1' # @param ['None', 0, 1, 2, 3, 4, 5]
third_channel = '2' # @param ['None', 0, 1, 2, 3, 4, 5]
# for nucleus segmentation
first_channel = '0' # @param ['None', 0, 1, 2, 3, 4, 5]
second_channel = '0' # @param ['None', 0, 1, 2, 3, 4, 5]
third_channel = '0' # @param ['None', 0, 1, 2, 3, 4, 5]

In [ ]:
selected_channels = []
for i, c in enumerate([first_channel, second_channel, third_channel]):
  if c == 'None':
    continue
  if int(c) > img.shape[-1]:
    assert False, 'invalid channel index, must have index greater or equal to the number of channels'
  if c != 'None':
    selected_channels.append(int(c))



img_selected_channels = np.zeros_like(img)
img_selected_channels[:, :, :len(selected_channels)] = img[:, :, selected_channels]


flow_threshold = 0.4
cellprob_threshold = 0.0
tile_norm_blocksize = 0

masks, flows, styles = model.eval(img_selected_channels, niter=250, flow_threshold=flow_threshold, cellprob_threshold=cellprob_threshold)

fig = plt.figure(figsize=(40,10))
plot.show_segmentation(fig, img_selected_channels, masks, flows[0])
plt.tight_layout()
plt.show()


In [ ]:
target_size = (img_.shape[1], img_.shape[0])
print(target_size)
if (target_size[0]!=img.shape[1] or target_size[1]!=img.shape[0]):
    # scale it back to keep the orignal size
    masks_rsz_ = cv2.resize(masks.astype('uint16'), target_size, interpolation=cv2.INTER_NEAREST).astype('uint16')
    masks_rsz = masks_rsz_
else:
    masks_rsz = masks.copy()

# plt.imshow(img_)
# plt.show()

# plt.imshow(masks_rsz)
# plt.show()

# make as a grid of images
plt.figure(figsize=(30, 20))
plt.subplot(1, 3, 1)
plt.imshow(img_selected_channels)
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(masks)
plt.axis('off')

plt.subplot(1, 3, 3)
outline = plot.outline_view(img_selected_channels, masks, color=[0, 1, 0], mode="inner")
plt.imshow(outline)
plt.axis('off')
plt.show()

MASK_SAVE = False
if MASK_SAVE:
    masks_ext = ".png" if image_ext == ".png" else ".tif"
    io.imsave(dir / (files[image_no].stem + "_masks" + masks_ext), masks_rsz)


In [ ]:
import os
dir_measured_cells = "/home/kargin/Projects/repositories/cellpose/images/measured-cells"
cell_name = "AH_027_11_10-measured2"
cell_dir = os.path.join(dir_measured_cells, cell_name + "_RGB.tif")
img_ = io.imread(cell_dir)
####
masks_ = masks_rsz.copy()
# masks_no_smooth = masks_rsz_.copy()
with_image, only_outline = outline_view_skimage(img_, masks_, color=[255, 255, 255], mode="inner")
plt.imshow(with_image)
plt.axis('off')
plt.imsave(f'with_image_{cell_name}.png', with_image)
# with_image, only_outline = outline_view_skimage(with_image, masks_no_smooth, color=[0, 0, 255], mode="inner")
# plt.show()
# plt.imsave(f'with_image_no_smooth_{cell_name}.png', with_image)
# with_image, only_outline = outline_view_skimage(img_, masks_, color=[255, 255, 255], mode="inner")
# plt.imshow(only_outline, cmap='binary')
# plt.axis('off')
# # save the image
# plt.imsave(f'only_outline_{cell_name}.png', only_outline, cmap='binary')
# plt.show()

## Plot masks with cell numbers

This will help you identify which cell you want to analyze by showing the mask number on each cell.


In [ ]:
from scipy.ndimage import find_objects, center_of_mass

def plot_masks_with_labels(img, masks, figsize=(20, 15), fontsize=10):
    """
    Plot masks overlaid on image with cell numbers labeled on each cell.
    
    Args:
        img: Image to display (2D or 3D array)
        masks: Mask array from Cellpose (0=background, 1,2,...=cell labels)
        figsize: Figure size tuple (width, height)
        fontsize: Font size for labels
    """
    # Create mask overlay
    overlay = plot.mask_overlay(img, masks)
    
    # Calculate centroids for each mask
    slices = find_objects(masks)
    centroids = []
    labels = []
    
    for i, slc in enumerate(slices):
        if slc is not None:
            # Get the mask for this cell (cell label is i+1)
            mask_label = i + 1
            # Extract the slice
            mask_slice = masks[slc] == mask_label
            if mask_slice.sum() > 0:
                # Calculate center of mass
                com = center_of_mass(mask_slice)
                # Adjust coordinates to full image coordinates
                y_cent = int(com[0] + slc[0].start)
                x_cent = int(com[1] + slc[1].start)
                centroids.append((y_cent, x_cent))
                labels.append(mask_label)
    
    # Plot
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(overlay)
    num_cells = masks.max() if masks.max() > 0 else 0
    ax.set_title(f'Masks with labels (total cells: {num_cells})', fontsize=16)
    ax.axis('off')
    
    # Add text labels at centroids
    for (y, x), label in zip(centroids, labels):
        ax.text(x, y, str(label), 
                fontsize=fontsize, fontweight='bold',
                color='white', 
                bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.7, edgecolor='white', linewidth=1.5),
                ha='center', va='center')
    
    plt.tight_layout()
    plt.show()
    
    return fig, ax

In [ ]:
# # Plot masks with labels (using resized masks that match original image size)
# print("Plotting masks with labels on original image size...")
# plot_masks_with_labels(img_, masks_rsz, figsize=(20, 15))

# You can also plot the resized version used for segmentation
print("\nPlotting masks with labels on resized image (used for segmentation)...")
plot_masks_with_labels(img_selected_channels, masks, figsize=(15, 10))


## Finding cell statistics

In [ ]:
# Get number of cells
num_cells = masks_rsz.max()

# Get pixels belonging to cell #
cell_number = 22 # 1 7 22 
cell_mask = (masks_rsz == cell_number)

# Get area of each cell
unique_labels, counts = np.unique(masks_rsz, return_counts=True)
cell_areas = dict(zip(unique_labels[1:], counts[1:]))  # Exclude 0 (background)

print('Cell area of chosen cell:', cell_areas[cell_number], 'pixels')

# plot cell 5 area
plt.imshow(cell_mask, cmap='binary')
plt.axis('off')
plt.show()

## Calculate cell area in square micrometers

Pixel to micrometer conversion: The biologist provided that 1 pixel = 0.100 micrometers.

Conversion factor for area: 1 pixel² = (0.100 μm)² = 0.01 μm²


In [ ]:
# Pixel to micrometer conversion factor
# 1 pixel = 0.100 micrometers (provided by biologist)
PIXEL_TO_MICROMETER = 0.039800  # micrometers per pixel - PhysicalSizeX="0.039800" PhysicalSizeY="0.039800"
PIXEL2_TO_MICROMETER2 = PIXEL_TO_MICROMETER ** 2  # 0.01 μm² per pixel²

def calculate_cell_area(masks, cell_number, pixel_to_um=PIXEL_TO_MICROMETER):
    """
    Calculate the area of a specific cell in pixels and square micrometers.
    
    Args:
        masks: Mask array from Cellpose (0=background, 1,2,...=cell labels)
        cell_number: Integer cell label (e.g., 1, 2, 3, ...)
        pixel_to_um: Conversion factor from pixels to micrometers (default: 0.100)
    
    Returns:
        dict: Dictionary with 'pixels' and 'micrometers_squared' keys
    """
    # Count pixels belonging to this cell
    cell_pixels = np.sum(masks == cell_number)
    
    # Calculate area in square micrometers
    area_um2 = cell_pixels * (pixel_to_um ** 2)
    
    return {
        'cell_number': cell_number,
        'area_pixels': cell_pixels,
        'area_um2': area_um2,
        'pixel_to_um': pixel_to_um
    }

def calculate_multiple_cell_areas(masks, cell_numbers, pixel_to_um=PIXEL_TO_MICROMETER):
    """
    Calculate areas for multiple cells at once.
    
    Args:
        masks: Mask array from Cellpose (0=background, 1,2,...=cell labels)
        cell_numbers: List of cell numbers (e.g., [1, 5, 12])
        pixel_to_um: Conversion factor from pixels to micrometers (default: 0.100)
    
    Returns:
        list: List of dictionaries with area information for each cell
    """
    results = []
    for cell_num in cell_numbers:
        if cell_num in masks:
            result = calculate_cell_area(masks, cell_num, pixel_to_um)
            results.append(result)
        else:
            print(f"Warning: Cell {cell_num} not found in masks (max cell number: {masks.max()})")
    return results

def print_cell_area(result):
    """Pretty print cell area information."""
    print(f"Cell {result['cell_number']}:")
    print(f"  Area in pixels: {result['area_pixels']:.2f} px²")
    print(f"  Area in square micrometers: {result['area_um2']:.2f} μm²")
    print()

In [ ]:
# Example usage: Calculate area for a specific cell
# Replace it with the cell number you want to analyze
cell_number = 49  # Change this to the cell number you identified from the labeled plot

if cell_number <= masks_rsz.max():
    result = calculate_cell_area(masks_rsz, cell_number, PIXEL_TO_MICROMETER)
    print_cell_area(result)
else:
    print(f"Cell {cell_number} does not exist. Max cell number is {masks_rsz.max()}")


In [ ]:
# Calculate areas for multiple cells at once
# Example: analyze cells 5, 10, 15, 20
cells_to_analyze = [1, 8, 41]  # Change this list to your desired cell numbers

print(f"Calculating areas for cells: {cells_to_analyze}")
print(f"Conversion: 1 pixel = {PIXEL_TO_MICROMETER} μm, so 1 px² = {PIXEL2_TO_MICROMETER2} μm²\n")
print("-" * 50)

results = calculate_multiple_cell_areas(masks_rsz, cells_to_analyze, PIXEL_TO_MICROMETER)

for result in results:
    print_cell_area(result)


In [ ]:
# Calculate area statistics for all cells
print("Area statistics for all detected cells:")
print(f"Conversion: 1 pixel = {PIXEL_TO_MICROMETER} μm, so 1 px² = {PIXEL2_TO_MICROMETER2} μm²\n")

unique_labels, pixel_counts = np.unique(masks_rsz, return_counts=True)
# Remove background (label 0)
cell_labels = unique_labels[1:]
cell_pixel_areas = pixel_counts[1:]

# Convert to square micrometers
cell_areas_um2 = cell_pixel_areas * PIXEL2_TO_MICROMETER2

# Create summary DataFrame-like output
print(f"{'Cell #':<10} {'Area (px²)':<15} {'Area (μm²)':<15}")
print("-" * 40)
for label, px_area, um2_area in zip(cell_labels, cell_pixel_areas, cell_areas_um2):
    print(f"{label:<10} {px_area:<15.2f} {um2_area:<15.2f}")

print("\nSummary statistics:")
print(f"Total number of cells: {len(cell_labels)}")
print(f"Mean area: {np.mean(cell_areas_um2):.2f} μm²")
print(f"Median area: {np.median(cell_areas_um2):.2f} μm²")
print(f"Min area: {np.min(cell_areas_um2):.2f} μm² (Cell {cell_labels[np.argmin(cell_areas_um2)]})")
print(f"Max area: {np.max(cell_areas_um2):.2f} μm² (Cell {cell_labels[np.argmax(cell_areas_um2)]})")
print(f"Std deviation: {np.std(cell_areas_um2):.2f} μm²")


## Run Cellpose-SAM on folder of images

if you have many large images, you may want to run them as a loop over images



In [ ]:
masks_ext = ".png" if image_ext == ".png" else ".tif"
for i in range(len(files)):
    f = files[i]
    img_ = io.imread(f)
    img = img_.copy()
    img = image_resize(img, resize = 1000)

    img = normalize99(img)
    img = np.clip(img, 0, 1)
    masks, flows, styles = model.eval(img, batch_size=32, flow_threshold=flow_threshold, cellprob_threshold=cellprob_threshold)
    fig = plt.figure(figsize=(24,10))
    plot.show_segmentation(fig, img, masks, flows[0])
    plt.tight_layout()
    plt.show()
    # io.imsave(dir / (f.stem + "_masks" + masks_ext), masks)
    

if you have small images, you may want to load all of them first and then run, so that they can be batched together on the GPU

In [ ]:
print("loading images")
imgs = [io.imread(files[i]) for i in trange(len(files))]

print("running cellpose-SAM")
masks, flows, styles = model.eval(imgs, batch_size=32, flow_threshold=flow_threshold, cellprob_threshold=cellprob_threshold,
                                  normalize={"tile_norm_blocksize": tile_norm_blocksize})

print("saving masks")
for i in trange(len(files)):
    f = files[i]
    io.imsave(dir / (f.stem + "_masks" + masks_ext), masks[i])

to save your masks for ImageJ, run the following code:

In [ ]:
for i in trange(len(files)):
    f = files[i]
    masks0 = io.imsave(dir / (f.name + "_masks" + masks_ext))
    io.save_rois(masks0, f)

## Shadow Analysis for Preprocessing Optimization

This section analyzes the shadow characteristics in the images to help design preprocessing 
techniques that reduce shadow artifacts and bring AI segmentation closer to human annotations.


In [ ]:
### Load and visualize one image with its channels
import numpy as np
import matplotlib.pyplot as plt
from cellpose import io
from skimage import color, filters, exposure
from scipy import ndimage
import cv2

# Load an example image
img_path = "/home/kargin/Projects/repositories/cellpose/images/AH_027_01_8.tif"
img_ = io.imread(img_path)
img_original = img_.copy()
img_original = image_resize(img_original, resize = 1000)

print(f"Image shape: {img_original.shape}")
print(f"Image dtype: {img_original.dtype}")
print(f"Image range: [{img_original.min()}, {img_original.max()}]")

# Normalize for display
if img_original.dtype == np.uint16:
    img_display = (img_original / img_original.max() * 255).astype(np.uint8)
else:
    img_display = img_original.copy()

# Display the original image and its channels
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Original image
axes[0, 0].imshow(img_display)
axes[0, 0].set_title('Original Image', fontsize=14)
axes[0, 0].axis('off')

# Individual channels
channel_names = ['Red Channel', 'Green Channel', 'Blue Channel']
for i in range(3):
    axes[0, i].imshow(img_display[:, :, i], cmap='gray')
    axes[0, i].set_title(f'{channel_names[i]}', fontsize=14)
    axes[0, i].axis('off')

# Convert to different color spaces for analysis
# LAB color space - L channel shows luminance (affected by shadows)
img_lab = color.rgb2lab(img_display)
axes[1, 0].imshow(img_lab[:, :, 0], cmap='gray')
axes[1, 0].set_title('LAB - L Channel (Luminance)', fontsize=14)
axes[1, 0].axis('off')

# HSV - V channel shows value/brightness
img_hsv = color.rgb2hsv(img_display)
axes[1, 1].imshow(img_hsv[:, :, 2], cmap='gray')
axes[1, 1].set_title('HSV - V Channel (Value/Brightness)', fontsize=14)
axes[1, 1].axis('off')

# Grayscale
img_gray = color.rgb2gray(img_display)
axes[1, 2].imshow(img_gray, cmap='gray')
axes[1, 2].set_title('Grayscale', fontsize=14)
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()


### Intensity Profile Analysis

Draw a line across a cell to analyze the intensity profile from background → shadow → cell → shadow → background.
This helps understand the shadow characteristics.


In [ ]:
from skimage.measure import profile_line

# Select a region with a clear cell - adjust these coordinates based on your image
# You can identify coordinates by looking at the image and finding a cell
# These are example coordinates - adjust based on where cells are in your image

# Let's first show a cropped region to identify good coordinates
# Crop a region that likely contains cells
crop_y_start, crop_y_end = 0, img_display.shape[0]
crop_x_start, crop_x_end = 0, img_display.shape[1]

img_crop = img_display[crop_y_start:crop_y_end, crop_x_start:crop_x_end]

plt.figure(figsize=(15, 10))
plt.imshow(img_crop)
plt.title('Cropped region - identify a cell to analyze', fontsize=14)
plt.colorbar(label='Intensity')

# Draw grid lines to help identify coordinates
ax = plt.gca()
ax.set_xticks(np.arange(0, img_crop.shape[1], 100))
ax.set_yticks(np.arange(0, img_crop.shape[0], 100))
ax.grid(True, alpha=0.3)
plt.show()

print(f"Cropped image shape: {img_crop.shape}")
print("Note: Coordinates shown are relative to the cropped image")


In [ ]:
# *** ADJUST THESE COORDINATES ***
# Draw a line from outside the cell, through the cell, to the other side
# Format: (row, col) - where row is Y and col is X in the cropped image
# Example line coordinates (adjust these based on where you see a cell):
line_start = (400, 340)   # (y, x) - start point (outside cell, in background)
line_end = (650, 500)     # (y, x) - end point (outside cell, on other side)

# Extract intensity profiles along the line for each channel
img_crop_gray = color.rgb2gray(img_crop)
img_crop_lab = color.rgb2lab(img_crop)

# Get profiles
profile_r = profile_line(img_crop[:, :, 0], line_start, line_end, mode='constant')
profile_g = profile_line(img_crop[:, :, 1], line_start, line_end, mode='constant')
profile_b = profile_line(img_crop[:, :, 2], line_start, line_end, mode='constant')
profile_gray = profile_line(img_crop_gray, line_start, line_end, mode='constant')
profile_L = profile_line(img_crop_lab[:, :, 0], line_start, line_end, mode='constant')

# Calculate gradient (edge detection along the profile)
gradient_gray = np.gradient(profile_gray)

# Plot the image with the line and the intensity profiles
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Show image with line
axes[0, 0].imshow(img_crop)
axes[0, 0].plot([line_start[1], line_end[1]], [line_start[0], line_end[0]], 'r-', linewidth=2, label='Profile line')
axes[0, 0].scatter([line_start[1], line_end[1]], [line_start[0], line_end[0]], c='yellow', s=100, zorder=5)
axes[0, 0].set_title('Image with profile line (red)', fontsize=14)
axes[0, 0].legend()
axes[0, 0].axis('off')

# Plot RGB channel profiles
axes[0, 1].plot(profile_r, 'r-', label='Red', alpha=0.7)
axes[0, 1].plot(profile_g, 'g-', label='Green', alpha=0.7)
axes[0, 1].plot(profile_b, 'b-', label='Blue', alpha=0.7)
axes[0, 1].set_xlabel('Position along line (pixels)')
axes[0, 1].set_ylabel('Intensity')
axes[0, 1].set_title('RGB Channel Intensity Profiles', fontsize=14)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot grayscale and LAB-L profiles
axes[1, 0].plot(profile_gray * 255, 'k-', label='Grayscale', linewidth=2)
axes[1, 0].plot(profile_L, 'purple', label='LAB-L (Luminance)', linewidth=2, linestyle='--')
axes[1, 0].set_xlabel('Position along line (pixels)')
axes[1, 0].set_ylabel('Intensity')
axes[1, 0].set_title('Grayscale & Luminance Profiles', fontsize=14)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot gradient (to see where edges are)
axes[1, 1].plot(gradient_gray * 255, 'orange', linewidth=2)
axes[1, 1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[1, 1].set_xlabel('Position along line (pixels)')
axes[1, 1].set_ylabel('Gradient (rate of change)')
axes[1, 1].set_title('Intensity Gradient (Edge Detection)', fontsize=14)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print analysis
print("\n=== Shadow Analysis ===")
print(f"Profile length: {len(profile_gray)} pixels")
print(f"Grayscale intensity range: [{profile_gray.min()*255:.1f}, {profile_gray.max()*255:.1f}]")
print(f"Max gradient magnitude: {np.abs(gradient_gray).max()*255:.2f}")
print("\nLook for:")
print("- Gradual intensity changes at cell boundaries = shadows")
print("- Sharp intensity changes = actual cell membrane")


### Test Preprocessing Techniques for Shadow Reduction

Let's compare several preprocessing techniques to see which one best reduces shadows while preserving cell boundaries.


In [ ]:
from skimage import morphology, restoration
from skimage.filters import unsharp_mask
from scipy.ndimage import gaussian_filter, median_filter

def rolling_ball_background(img, radius=50):
    """
    Rolling ball background subtraction - classic ImageJ approach.
    Works by estimating the background with morphological opening.
    """
    if img.ndim == 3:
        result = np.zeros_like(img, dtype=np.float64)
        for c in range(img.shape[2]):
            # Use white tophat (removes dark features smaller than structuring element)
            selem = morphology.disk(radius)
            result[:, :, c] = morphology.white_tophat(img[:, :, c], selem)
        return result
    else:
        selem = morphology.disk(radius)
        return morphology.white_tophat(img, selem)

def morphological_background_subtraction(img, disk_size=50):
    """
    Estimate and subtract background using morphological opening.
    """
    if img.ndim == 3:
        result = np.zeros_like(img, dtype=np.float64)
        for c in range(img.shape[2]):
            selem = morphology.disk(disk_size)
            background = morphology.opening(img[:, :, c], selem)
            result[:, :, c] = img[:, :, c].astype(np.float64) - background.astype(np.float64)
        result = np.clip(result, 0, 255)
        return result.astype(np.uint8)
    else:
        selem = morphology.disk(disk_size)
        background = morphology.opening(img, selem)
        result = img.astype(np.float64) - background.astype(np.float64)
        return np.clip(result, 0, 255).astype(np.uint8)

def gaussian_background_subtraction(img, sigma=50):
    """
    Subtract Gaussian-blurred background to remove low-frequency shadows.
    """
    if img.ndim == 3:
        result = np.zeros_like(img, dtype=np.float64)
        for c in range(img.shape[2]):
            background = gaussian_filter(img[:, :, c].astype(np.float64), sigma=sigma)
            result[:, :, c] = img[:, :, c].astype(np.float64) - background + 128  # Add offset to keep in range
        result = np.clip(result, 0, 255)
        return result.astype(np.uint8)
    else:
        background = gaussian_filter(img.astype(np.float64), sigma=sigma)
        result = img.astype(np.float64) - background + 128
        return np.clip(result, 0, 255).astype(np.uint8)

def clahe_enhancement(img, clip_limit=2.0, grid_size=8):
    """
    Apply CLAHE (Contrast Limited Adaptive Histogram Equalization).
    """
    if img.ndim == 3:
        # Convert to LAB, apply CLAHE to L channel
        img_lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(grid_size, grid_size))
        img_lab[:, :, 0] = clahe.apply(img_lab[:, :, 0])
        return cv2.cvtColor(img_lab, cv2.COLOR_LAB2RGB)
    else:
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(grid_size, grid_size))
        return clahe.apply(img)

def lab_luminance_normalization(img, percentile_low=2, percentile_high=98):
    """
    Normalize the luminance channel in LAB color space to reduce shadows.
    """
    img_lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB).astype(np.float64)
    L = img_lab[:, :, 0]
    
    # Normalize L channel using percentiles
    p_low = np.percentile(L, percentile_low)
    p_high = np.percentile(L, percentile_high)
    L_norm = (L - p_low) / (p_high - p_low + 1e-10) * 255
    L_norm = np.clip(L_norm, 0, 255)
    
    img_lab[:, :, 0] = L_norm
    return cv2.cvtColor(img_lab.astype(np.uint8), cv2.COLOR_LAB2RGB)

def edge_enhance(img, amount=1.5, sigma=1.0):
    """
    Apply unsharp masking to enhance edges.
    """
    if img.ndim == 3:
        result = np.zeros_like(img, dtype=np.float64)
        for c in range(img.shape[2]):
            result[:, :, c] = unsharp_mask(img[:, :, c], radius=sigma, amount=amount)
        return (result * 255).astype(np.uint8)
    else:
        return (unsharp_mask(img, radius=sigma, amount=amount) * 255).astype(np.uint8)

print("Preprocessing functions defined successfully!")


In [ ]:
# Apply all preprocessing techniques and compare
# Use the cropped region for faster processing and visualization

print("Applying preprocessing techniques...")

# Original
img_orig = img_crop.copy()

# 1. Morphological background subtraction
print("1. Morphological background subtraction...")
img_morph = morphological_background_subtraction(img_crop, disk_size=30)

# 2. Gaussian background subtraction
print("2. Gaussian background subtraction...")
img_gauss = gaussian_background_subtraction(img_crop, sigma=50)

# 3. CLAHE
print("3. CLAHE enhancement...")
img_clahe = clahe_enhancement(img_crop, clip_limit=3.0, grid_size=8)

# 4. LAB luminance normalization
print("4. LAB luminance normalization...")
img_lab_norm = lab_luminance_normalization(img_crop)

# 5. Edge enhancement (unsharp mask)
print("5. Edge enhancement...")
img_edge = edge_enhance(img_crop, amount=2.0, sigma=2.0)

# 6. Combined: Gaussian background subtraction + CLAHE
print("6. Combined: Gaussian + CLAHE...")
img_combined = gaussian_background_subtraction(img_crop, sigma=40)
img_combined = clahe_enhancement(img_combined, clip_limit=2.0, grid_size=8)

print("Done!")

# Visualize all results
fig, axes = plt.subplots(2, 4, figsize=(24, 12))

titles = [
    'Original',
    'Morphological BG Sub\n(disk=30)',
    'Gaussian BG Sub\n(sigma=50)',
    'CLAHE\n(clip=3.0)',
    'LAB Luminance Norm',
    'Edge Enhancement\n(unsharp mask)',
    'Combined:\nGauss BG + CLAHE',
    ''
]

images = [img_orig, img_morph, img_gauss, img_clahe, img_lab_norm, img_edge, img_combined, None]

for idx, (ax, img, title) in enumerate(zip(axes.flat, images, titles)):
    if img is not None:
        ax.imshow(img)
        ax.set_title(title, fontsize=12)
    ax.axis('off')

# Hide the last empty subplot
axes[1, 3].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# Compare intensity profiles across all preprocessing methods
# This will show us which method best reduces the shadow gradient

# Convert all to grayscale for profile comparison
def to_gray(img):
    if img.ndim == 3:
        return color.rgb2gray(img)
    return img

profiles = {
    'Original': to_gray(img_orig),
    'Morph BG Sub': to_gray(img_morph),
    'Gauss BG Sub': to_gray(img_gauss),
    'CLAHE': to_gray(img_clahe),
    'LAB Norm': to_gray(img_lab_norm),
    'Edge Enhance': to_gray(img_edge),
    'Combined': to_gray(img_combined),
}

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot all intensity profiles
colors_plot = ['black', 'red', 'blue', 'green', 'purple', 'orange', 'brown']
for (name, img_gray), col in zip(profiles.items(), colors_plot):
    profile = profile_line(img_gray, line_start, line_end, mode='constant')
    # Normalize to 0-1 range for comparison
    profile_norm = (profile - profile.min()) / (profile.max() - profile.min() + 1e-10)
    axes[0].plot(profile_norm, label=name, color=col, alpha=0.8, linewidth=2 if name == 'Original' else 1.5)

axes[0].set_xlabel('Position along line (pixels)', fontsize=12)
axes[0].set_ylabel('Normalized Intensity', fontsize=12)
axes[0].set_title('Intensity Profiles Comparison (Normalized)', fontsize=14)
axes[0].legend(loc='best')
axes[0].grid(True, alpha=0.3)

# Plot gradients (edge sharpness)
for (name, img_gray), col in zip(profiles.items(), colors_plot):
    profile = profile_line(img_gray, line_start, line_end, mode='constant')
    gradient = np.abs(np.gradient(profile))
    axes[1].plot(gradient, label=name, color=col, alpha=0.8, linewidth=2 if name == 'Original' else 1.5)

axes[1].set_xlabel('Position along line (pixels)', fontsize=12)
axes[1].set_ylabel('Gradient Magnitude', fontsize=12)
axes[1].set_title('Edge Sharpness Comparison (|gradient|)', fontsize=14)
axes[1].legend(loc='best')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics
print("\n=== Edge Sharpness Analysis ===")
print("Higher max gradient = sharper edges (good for cell boundary detection)")
print("Lower gradient in 'shadow' regions = less shadow effect\n")

for name, img_gray in profiles.items():
    profile = profile_line(img_gray, line_start, line_end, mode='constant')
    gradient = np.abs(np.gradient(profile))
    print(f"{name:20s}: Max gradient = {gradient.max():.4f}, Mean gradient = {gradient.mean():.4f}")


### Run Segmentation on Preprocessed Images

Now let's run Cellpose-SAM on the original and preprocessed images to compare how preprocessing affects segmentation boundaries.


In [ ]:
# Run segmentation on original and selected preprocessed images
# We'll compare the mask boundaries

from cellpose import models, plot

# Reload model if needed
try:
    model
except NameError:
    model = models.CellposeModel(gpu=True)

# Prepare images for segmentation (need to be float 0-1 or properly normalized)
def prepare_for_segmentation(img):
    """Normalize image for cellpose segmentation."""
    img_float = img.astype(np.float32)
    # Percentile normalization
    p1, p99 = np.percentile(img_float, [1, 99])
    img_norm = (img_float - p1) / (p99 - p1 + 1e-10)
    return np.clip(img_norm, 0, 1)

# Select the most promising preprocessing methods to test
test_images = {
    'Original': img_crop,
    'Morph BG Sub': img_morph,
    'Gauss BG Sub': img_gauss,
    'CLAHE': img_clahe,
    'LAB Norm': img_lab_norm,
    'Edge Enhance': img_edge,
    'Combined': img_combined,
}

# Run segmentation on each
segmentation_results = {}
flow_threshold = 0.4
cellprob_threshold = 0.0

print("Running segmentation on different preprocessing methods...")
for name, img in test_images.items():
    print(f"  Segmenting: {name}...")
    img_prep = prepare_for_segmentation(img)
    masks, flows, styles = model.eval(img_prep, niter=250, 
                                       flow_threshold=flow_threshold, 
                                       cellprob_threshold=cellprob_threshold)
    segmentation_results[name] = {
        'img': img,
        'img_prep': img_prep,
        'masks': masks,
        'flows': flows
    }
    print(f"    Found {masks.max()} cells")

print("Done!")


In [ ]:
# Visualize segmentation results comparison
from skimage.segmentation import find_boundaries

fig, axes = plt.subplots(2, 7, figsize=(24, 12))

for idx, (name, results) in enumerate(segmentation_results.items()):
    img = results['img']
    masks = results['masks']
    
    # Top row: Image with mask overlay
    overlay = plot.mask_overlay(img, masks)
    axes[0, idx].imshow(overlay)
    axes[0, idx].set_title(f'{name}\n({masks.max()} cells)', fontsize=12)
    axes[0, idx].axis('off')
    
    # Bottom row: Image with outline only
    outlines = find_boundaries(masks, mode='inner')
    img_with_outline = img.copy().astype(np.float32)
    if img_with_outline.max() > 1:
        img_with_outline = img_with_outline / 255.0
    
    # Make outline visible
    for c in range(3):
        img_with_outline[:, :, c][outlines] = [0, 1, 0][c]  # Green outlines
    
    axes[1, idx].imshow(img_with_outline)
    axes[1, idx].set_title(f'{name} - Outlines', fontsize=12)
    axes[1, idx].axis('off')

plt.suptitle('Segmentation Comparison: Original vs Preprocessed Images', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# Print cell count comparison
print("\n=== Cell Count Comparison ===")
for name, results in segmentation_results.items():
    print(f"{name:20s}: {results['masks'].max()} cells detected")


In [ ]:
# Compare cell areas between original and preprocessed segmentations
# Smaller areas (while maintaining correct cell count) suggests tighter boundaries

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

all_areas = {}
for name, results in segmentation_results.items():
    masks = results['masks']
    if masks.max() > 0:
        unique_labels, counts = np.unique(masks, return_counts=True)
        # Exclude background (label 0)
        cell_areas = counts[1:] if unique_labels[0] == 0 else counts
        all_areas[name] = cell_areas

# Box plot comparison
axes[0].boxplot([all_areas[name] for name in all_areas.keys()], 
                labels=list(all_areas.keys()))
axes[0].set_ylabel('Cell Area (pixels)', fontsize=12)
axes[0].set_title('Cell Area Distribution Comparison', fontsize=14)
axes[0].tick_params(axis='x', rotation=45)

# Histogram overlay
for name, areas in all_areas.items():
    axes[1].hist(areas, bins=30, alpha=0.5, label=name)
axes[1].set_xlabel('Cell Area (pixels)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Cell Area Histogram', fontsize=14)
axes[1].legend()

plt.tight_layout()
plt.show()

# Print statistics
print("\n=== Cell Area Statistics (in pixels) ===")
print(f"{'Method':<20} {'Mean':<12} {'Median':<12} {'Std':<12} {'Min':<12} {'Max':<12}")
print("-" * 80)
for name, areas in all_areas.items():
    print(f"{name:<20} {np.mean(areas):<12.1f} {np.median(areas):<12.1f} {np.std(areas):<12.1f} {np.min(areas):<12.1f} {np.max(areas):<12.1f}")

# Calculate area reduction compared to original
print("\n=== Area Reduction vs Original ===")
orig_mean = np.mean(all_areas.get('Original', [0]))
for name, areas in all_areas.items():
    if name != 'Original' and orig_mean > 0:
        reduction = (orig_mean - np.mean(areas)) / orig_mean * 100
        print(f"{name:<20}: {reduction:+.1f}% change in mean area")


### Summary and Recommendations

After running the analysis above, evaluate which preprocessing method:

1. **Reduces mean cell area** - Indicates tighter boundaries (closer to human annotations)
2. **Maintains correct cell count** - Doesn't merge or split cells incorrectly  
3. **Shows sharper edges in the intensity profile** - Less shadow gradient

**Promising approaches based on typical results:**

- **Gaussian Background Subtraction**: Good for removing low-frequency illumination variations and shadows
- **CLAHE**: Enhances local contrast, can help distinguish cell membrane from shadow
- **Combined (Gauss + CLAHE)**: Often the best balance of shadow removal and edge enhancement

**Next steps:**
1. Run the cells above to see results on your specific images
2. Identify which preprocessing method works best
3. Apply that method before segmentation in your pipeline
4. Fine-tune parameters (sigma, clip_limit, etc.) based on results


## Channel Analysis: Why Removing RED Channel Helps

Investigation into why using only Green+Blue channels (removing Red) produces segmentation closer to human annotations.


In [ ]:
# Analyze why Red channel captures shadows while Green/Blue don't

# Use the cropped region from earlier analysis
# If not defined, load it
try:
    img_crop
except NameError:
    img_path = "/home/kargin/Projects/repositories/cellpose/images/AH_027_01_10.tif"
    img_original = io.imread(img_path)
    if img_original.dtype == np.uint16:
        img_display = (img_original / img_original.max() * 255).astype(np.uint8)
    else:
        img_display = img_original.copy()
    crop_y_start, crop_y_end = 1500, 2500
    crop_x_start, crop_x_end = 2000, 3500
    img_crop = img_display[crop_y_start:crop_y_end, crop_x_start:crop_x_end]

# Split channels
red_channel = img_crop[:, :, 0]
green_channel = img_crop[:, :, 1]
blue_channel = img_crop[:, :, 2]

# Calculate edge strength (gradient magnitude) for each channel
from scipy.ndimage import sobel

def edge_strength(channel):
    """Calculate edge strength using Sobel operator."""
    sx = sobel(channel.astype(float), axis=0)
    sy = sobel(channel.astype(float), axis=1)
    return np.sqrt(sx**2 + sy**2)

edge_red = edge_strength(red_channel)
edge_green = edge_strength(green_channel)
edge_blue = edge_strength(blue_channel)

# Visualize channels and their edge maps
fig, axes = plt.subplots(3, 3, figsize=(18, 18))

# Row 1: Original channels
axes[0, 0].imshow(red_channel, cmap='Reds')
axes[0, 0].set_title('RED Channel', fontsize=14)
axes[0, 0].axis('off')

axes[0, 1].imshow(green_channel, cmap='Greens')
axes[0, 1].set_title('GREEN Channel', fontsize=14)
axes[0, 1].axis('off')

axes[0, 2].imshow(blue_channel, cmap='Blues')
axes[0, 2].set_title('BLUE Channel', fontsize=14)
axes[0, 2].axis('off')

# Row 2: Edge maps
axes[1, 0].imshow(edge_red, cmap='hot')
axes[1, 0].set_title(f'RED Edges\n(mean={edge_red.mean():.2f})', fontsize=14)
axes[1, 0].axis('off')

axes[1, 1].imshow(edge_green, cmap='hot')
axes[1, 1].set_title(f'GREEN Edges\n(mean={edge_green.mean():.2f})', fontsize=14)
axes[1, 1].axis('off')

axes[1, 2].imshow(edge_blue, cmap='hot')
axes[1, 2].set_title(f'BLUE Edges\n(mean={edge_blue.mean():.2f})', fontsize=14)
axes[1, 2].axis('off')

# Row 3: Histogram comparison
axes[2, 0].hist(red_channel.ravel(), bins=100, color='red', alpha=0.7, label='Red')
axes[2, 0].hist(green_channel.ravel(), bins=100, color='green', alpha=0.7, label='Green')
axes[2, 0].hist(blue_channel.ravel(), bins=100, color='blue', alpha=0.7, label='Blue')
axes[2, 0].set_xlabel('Intensity')
axes[2, 0].set_ylabel('Frequency')
axes[2, 0].set_title('Intensity Distribution by Channel', fontsize=14)
axes[2, 0].legend()

# Edge histogram
axes[2, 1].hist(edge_red.ravel(), bins=100, color='red', alpha=0.7, label='Red')
axes[2, 1].hist(edge_green.ravel(), bins=100, color='green', alpha=0.7, label='Green')
axes[2, 1].hist(edge_blue.ravel(), bins=100, color='blue', alpha=0.7, label='Blue')
axes[2, 1].set_xlabel('Edge Strength')
axes[2, 1].set_ylabel('Frequency')
axes[2, 1].set_title('Edge Strength Distribution by Channel', fontsize=14)
axes[2, 1].legend()
axes[2, 1].set_xlim([0, 100])  # Focus on relevant range

# Channel correlation
axes[2, 2].text(0.1, 0.8, "Channel Statistics:", fontsize=14, fontweight='bold', transform=axes[2, 2].transAxes)
axes[2, 2].text(0.1, 0.65, f"Red   - Mean: {red_channel.mean():.1f}, Std: {red_channel.std():.1f}", fontsize=12, transform=axes[2, 2].transAxes)
axes[2, 2].text(0.1, 0.50, f"Green - Mean: {green_channel.mean():.1f}, Std: {green_channel.std():.1f}", fontsize=12, transform=axes[2, 2].transAxes)
axes[2, 2].text(0.1, 0.35, f"Blue  - Mean: {blue_channel.mean():.1f}, Std: {blue_channel.std():.1f}", fontsize=12, transform=axes[2, 2].transAxes)
axes[2, 2].text(0.1, 0.15, f"\nEdge Sharpness (higher = sharper):", fontsize=12, fontweight='bold', transform=axes[2, 2].transAxes)
axes[2, 2].text(0.1, 0.0, f"Red: {edge_red.mean():.2f}  Green: {edge_green.mean():.2f}  Blue: {edge_blue.mean():.2f}", fontsize=12, transform=axes[2, 2].transAxes)
axes[2, 2].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# Compare intensity profiles across cell boundary for each channel
# This will show if Red channel has "blurrier" edges (more shadow)

from skimage.measure import profile_line

# Use the same line coordinates from earlier, or define new ones
try:
    line_start, line_end
except NameError:
    line_start = (500, 100)
    line_end = (500, 600)

# Get profiles for each channel
profile_red = profile_line(red_channel, line_start, line_end, mode='constant')
profile_green = profile_line(green_channel, line_start, line_end, mode='constant')
profile_blue = profile_line(blue_channel, line_start, line_end, mode='constant')

# Calculate gradients (edge sharpness)
grad_red = np.abs(np.gradient(profile_red))
grad_green = np.abs(np.gradient(profile_green))
grad_blue = np.abs(np.gradient(profile_blue))

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Show image with profile line
axes[0, 0].imshow(img_crop)
axes[0, 0].plot([line_start[1], line_end[1]], [line_start[0], line_end[0]], 'yellow', linewidth=3)
axes[0, 0].scatter([line_start[1], line_end[1]], [line_start[0], line_end[0]], c='white', s=100, zorder=5)
axes[0, 0].set_title('Profile Line Location', fontsize=14)
axes[0, 0].axis('off')

# Intensity profiles (normalized for comparison)
def normalize(arr):
    return (arr - arr.min()) / (arr.max() - arr.min() + 1e-10)

axes[0, 1].plot(normalize(profile_red), 'r-', linewidth=2, label='Red', alpha=0.8)
axes[0, 1].plot(normalize(profile_green), 'g-', linewidth=2, label='Green', alpha=0.8)
axes[0, 1].plot(normalize(profile_blue), 'b-', linewidth=2, label='Blue', alpha=0.8)
axes[0, 1].set_xlabel('Position along line (pixels)', fontsize=12)
axes[0, 1].set_ylabel('Normalized Intensity', fontsize=12)
axes[0, 1].set_title('Intensity Profile by Channel', fontsize=14)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Gradient (edge sharpness) comparison
axes[1, 0].plot(grad_red, 'r-', linewidth=2, label=f'Red (max={grad_red.max():.1f})', alpha=0.8)
axes[1, 0].plot(grad_green, 'g-', linewidth=2, label=f'Green (max={grad_green.max():.1f})', alpha=0.8)
axes[1, 0].plot(grad_blue, 'b-', linewidth=2, label=f'Blue (max={grad_blue.max():.1f})', alpha=0.8)
axes[1, 0].set_xlabel('Position along line (pixels)', fontsize=12)
axes[1, 0].set_ylabel('Gradient Magnitude', fontsize=12)
axes[1, 0].set_title('Edge Sharpness by Channel\n(Higher peaks = sharper edges)', fontsize=14)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Summary statistics
channel_stats = {
    'Red': {'profile': profile_red, 'gradient': grad_red},
    'Green': {'profile': profile_green, 'gradient': grad_green},
    'Blue': {'profile': profile_blue, 'gradient': grad_blue},
}

# Calculate "shadow width" - width of transition zone
def estimate_transition_width(gradient, threshold_pct=0.2):
    """Estimate the width of the transition zone (shadow region)."""
    max_grad = gradient.max()
    threshold = max_grad * threshold_pct
    above_threshold = gradient > threshold
    # Find runs of True values
    transitions = np.diff(above_threshold.astype(int))
    starts = np.where(transitions == 1)[0]
    ends = np.where(transitions == -1)[0]
    if len(starts) > 0 and len(ends) > 0:
        widths = []
        for s, e in zip(starts, ends):
            widths.append(e - s)
        return np.mean(widths) if widths else 0
    return 0

summary_text = "=== Channel Edge Analysis ===\n\n"
summary_text += f"{'Channel':<10} {'Max Grad':<12} {'Mean Grad':<12} {'Sharpness':<15}\n"
summary_text += "-" * 50 + "\n"

for ch_name, data in channel_stats.items():
    max_g = data['gradient'].max()
    mean_g = data['gradient'].mean()
    # Sharpness ratio: max / mean - higher means sharper edges vs smooth regions
    sharpness = max_g / (mean_g + 1e-10)
    summary_text += f"{ch_name:<10} {max_g:<12.2f} {mean_g:<12.4f} {sharpness:<15.2f}\n"

summary_text += "\nInterpretation:\n"
summary_text += "- Higher 'Sharpness' = more distinct cell boundaries\n"
summary_text += "- Lower sharpness in Red channel = more shadow/blur\n"
summary_text += "- This explains why removing Red improves segmentation!"

axes[1, 1].text(0.05, 0.95, summary_text, fontsize=11, family='monospace',
                verticalalignment='top', transform=axes[1, 1].transAxes,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

# Print analysis
print("\n" + "="*60)
print("KEY FINDING: Why Green+Blue works better than RGB")
print("="*60)
print(f"\nEdge Sharpness (max gradient):")
print(f"  Red:   {grad_red.max():.2f}")
print(f"  Green: {grad_green.max():.2f}")
print(f"  Blue:  {grad_blue.max():.2f}")
print(f"\nIf Green/Blue have higher max gradients, they have")
print(f"sharper cell boundaries with less shadow interference.")


In [ ]:
# Visualize the "shadow region" that Red channel captures but Green/Blue don't
# This is the region between where Red sees the cell edge vs where Green/Blue see it

# Create comparison images
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Original RGB
axes[0, 0].imshow(img_crop)
axes[0, 0].set_title('Original RGB', fontsize=14)
axes[0, 0].axis('off')

# Green + Blue only (what you're using now)
img_gb = np.zeros_like(img_crop)
img_gb[:, :, 1] = green_channel  # Green channel
img_gb[:, :, 2] = blue_channel   # Blue channel
axes[0, 1].imshow(img_gb)
axes[0, 1].set_title('Green + Blue Only\n(Your improved method)', fontsize=14)
axes[0, 1].axis('off')

# Red channel only
img_r = np.zeros_like(img_crop)
img_r[:, :, 0] = red_channel
axes[0, 2].imshow(img_r)
axes[0, 2].set_title('Red Channel Only\n(Contains shadow information)', fontsize=14)
axes[0, 2].axis('off')

# Difference: What Red "sees" that Green/Blue don't
# Normalize each channel first
red_norm = red_channel.astype(float) / (red_channel.max() + 1e-10)
green_norm = green_channel.astype(float) / (green_channel.max() + 1e-10)
blue_norm = blue_channel.astype(float) / (blue_channel.max() + 1e-10)

# Red minus average of Green and Blue - shows the "shadow excess" in red
shadow_excess = red_norm - (green_norm + blue_norm) / 2
shadow_excess = np.clip(shadow_excess, 0, 1)  # Keep only positive (where red > others)

axes[1, 0].imshow(shadow_excess, cmap='Reds')
axes[1, 0].set_title('Shadow Region\n(Red excess over Green/Blue)', fontsize=14)
axes[1, 0].axis('off')

# Edge comparison: overlay edges from different channels
edges_red = edge_red > np.percentile(edge_red, 95)
edges_green = edge_green > np.percentile(edge_green, 95)
edges_blue = edge_blue > np.percentile(edge_blue, 95)

# Create colored edge overlay
edge_overlay = np.zeros((*img_crop.shape[:2], 3), dtype=np.float32)
edge_overlay[:, :, 0] = edges_red.astype(float)   # Red edges in red
edge_overlay[:, :, 1] = edges_green.astype(float) # Green edges in green
edge_overlay[:, :, 2] = edges_blue.astype(float)  # Blue edges in blue

axes[1, 1].imshow(img_crop)
axes[1, 1].imshow(edge_overlay, alpha=0.7)
axes[1, 1].set_title('Edge Comparison Overlay\n(R=Red edges, G=Green edges, B=Blue edges)', fontsize=14)
axes[1, 1].axis('off')

# Where edges differ (Red edge but not Green/Blue = likely shadow boundary)
red_only_edges = edges_red & ~(edges_green | edges_blue)
axes[1, 2].imshow(img_crop)
axes[1, 2].imshow(np.dstack([red_only_edges, np.zeros_like(red_only_edges), np.zeros_like(red_only_edges)]).astype(np.float32), alpha=0.7)
axes[1, 2].set_title('Red-Only Edges\n(Shadow boundaries detected by Red)', fontsize=14)
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

# Count shadow pixels
shadow_pixels = np.sum(shadow_excess > 0.1)
total_pixels = shadow_excess.size
print(f"\n=== Shadow Analysis ===")
print(f"Pixels where Red significantly exceeds Green/Blue: {shadow_pixels:,} ({100*shadow_pixels/total_pixels:.1f}%)")
print(f"These are the 'shadow' regions that Red channel captures but G/B don't.")


### Conclusion: Why Green+Blue Works Better

Based on the analysis above, the Red channel likely captures **shadow/halo artifacts** that extend beyond the actual cell membrane because:

1. **Optical Properties**: Red light (longer wavelength ~620-700nm) scatters more diffusely than blue/green light, creating "fuzzy" edges
2. **Staining Chemistry**: If using eosin or similar red-absorbing stains, they can diffuse beyond cell boundaries
3. **Illumination Artifacts**: Shadows from uneven illumination often appear warmer (more red) in brightfield microscopy

**Recommendation**: Use `first_channel='1', second_channel='1', third_channel='2'` (Green+Blue only) for your segmentation pipeline. This is a simple but effective solution that requires no image preprocessing!
